# Exploração · `bronze.geracao_distribuida`

Notebook de **investigação**, fora do pipeline. Descobre os problemas de qualidade da tabela de geração distribuída (GD) e mede o que ela explica no diagnóstico da receita, antes de escrever a Silver.

**Por que esta tabela importa:** quando a UC gera a própria energia (painel solar), a energia gerada é abatida do consumo faturado. Se o número de UCs com GD cresceu entre o primeiro ano e os últimos 12 meses, isso ajuda a explicar a pergunta do diretor: **volume faturado −4% com a base de clientes estável**.


| # | Seção | Pergunta |
|---|---|---|
| 1 | Visão geral | Colunas, tipos e volume (o case fala em ~3,5 mil) |
| 2 | Nomes das colunas | Conferir os nomes reais antes de qualquer análise |
| 3 | Cabeçalho repetido | Há linha com o nome das colunas como dado? |
| 4 | Nulos e códigos de ausência | `N/A`, `NULL`, `-1`, `-`, texto vazio |
| 5 | Duplicatas | Cópias exatas? Mesma UC com mais de um registro? |
| 6 | Formatos | Padrão de cada coluna |
| 7 | Categorias | Grafias diferentes para o mesmo valor |
| 8 | Integridade com UCs | Órfãs? As 200 UCs de teste aparecem aqui? |
| 9 | Potência | Converte para número? Valores impossíveis? |
| 10 | Data de conexão | Formatos, ordem dia/mês, datas impossíveis |
| 11 | GD ao longo do tempo | Quantas UCs tinham GD no primeiro ano e nos últimos 12 meses |
| 12 | Impacto no volume | Quanto da queda de volume vem das UCs com GD |
| 13 | Resumo | Regras propostas para a Silver |

**Pré-requisitos:** `silver.unidades_consumidoras` e `silver.leituras` gravadas na sua conta (seções 8, 11 e 12).

In [0]:
from pyspark.sql import functions as F

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

gd_raw = spark.table(f"{bronze_schema}.geracao_distribuida")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def sem_acento(col):
    """upper + trim + sem acento + espaços simples: para comparar grafias."""
    c = F.translate(F.upper(F.trim(col)), "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")
    return F.regexp_replace(c, r"\s+", " ")

def padrao(col):
    """Troca dígitos por 9 e letras por A: mostra o formato do valor."""
    return F.regexp_replace(F.regexp_replace(col, "[0-9]", "9"), "[A-Za-zÀ-ÿ]", "A")

# Períodos do diagnóstico (contrato da Gold)
P0_INI, P0_FIM = "2022-07", "2023-06"   # primeiro ano
P1_INI, P1_FIM = "2025-07", "2026-06"   # últimos 12 meses

## 1. Visão geral

**Achado:** 3.510 linhas, como o case informa (~3,5 mil). Cinco colunas de origem, todas texto: `id_uc`, `potencia_kwp`, `data_conexao`, `modalidade` e `energia_injetada_kwh_media_mensal`.

In [0]:
gd_raw.printSchema()
print(f"{n_br(gd_raw.count())} linhas (o case fala em ~3,5 mil)")
display(gd_raw.limit(10))

## 2. Nomes das colunas

**Esta é a única célula que depende dos nomes das colunas.** Os nomes abaixo são os esperados. Compare com o `printSchema` da seção 1: se algum estiver diferente, **corrija aqui** e rode de novo. A célula para com erro se um nome não existir, para nenhuma análise rodar em cima de uma coluna errada.

Se a tabela tiver outras colunas importantes (por exemplo, energia injetada ou fonte de geração), acrescente-as em `extras`: elas entram automaticamente nas seções de nulos, formatos e categorias.

In [0]:
COL_UC = "id_uc"               # unidade consumidora
COL_POTENCIA = "potencia_kwp"  # potência instalada
COL_DATA = "data_conexao"      # data de conexão da GD à rede
COL_MODALIDADE = "modalidade"  # ex.: geração na própria UC, autoconsumo remoto
extras = []                    # outras colunas que valem análise

obrigatorias = [COL_UC, COL_POTENCIA, COL_DATA, COL_MODALIDADE] + extras
faltando = [c for c in obrigatorias if c not in gd_raw.columns]
if faltando:
    raise ValueError(f"Colunas não encontradas: {faltando}. Colunas reais: {gd_raw.columns}. Corrija os nomes nesta célula.")

cols_origem = [c for c in gd_raw.columns if c not in ("source_file", "ingestion_timestamp")]
print("Colunas confirmadas:", obrigatorias)
print("Todas as colunas de origem:", cols_origem)

## 3. Cabeçalho repetido

**O que explora:** linhas em que o valor da coluna é o próprio nome da coluna (aconteceu em `faturas` e `unidades_consumidoras`). A partir daqui, as análises usam `gd` (sem essas linhas).

**Achado:** nenhuma linha de cabeçalho repetido.

In [0]:
cab = gd_raw.filter(F.col(COL_UC) == COL_UC)
print("linhas de cabeçalho:", cab.count())
gd = gd_raw.filter(~F.coalesce(F.col(COL_UC) == COL_UC, F.lit(False)))
print("linhas sem cabeçalho:", n_br(gd.count()))

## 4. Nulos e códigos de ausência

**O que explora:** nulos reais e as formas de "vazio" que apareceram nas outras tabelas (`N/A`, `NULL`, `-1`, `-`, texto vazio e a data coringa `9999-12-31`).

**Achado:** nenhum nulo e nenhum código de ausência em nenhuma coluna.

In [0]:
codigos = ["N/A", "NULL", "-1", "-", "", "9999-12-31"]
linhas = []
for c in cols_origem:
    linha = {"coluna": c, "nulo_real": gd.filter(F.col(c).isNull()).count()}
    for cod in codigos:
        linha[f"'{cod}'"] = gd.filter(F.trim(F.col(c)) == cod).count()
    linhas.append(linha)
display(spark.createDataFrame(linhas))

## 5. Duplicatas

**O que explora:**
- **Cópias exatas** (linhas idênticas): removidas sem perda, como em medidores e leituras.
- **Mesma UC com mais de um registro e conteúdo diferente:** pode ser legítimo (a UC ampliou o sistema ou instalou um segundo) ou erro. Mostra quais colunas mudam entre as versões. Isso decide o grão da Silver: 1 linha por UC ou 1 linha por instalação.

**Achado:** 3.510 linhas, 3.510 linhas distintas e 3.510 `id_uc` distintos. Não há cópias, e **cada UC tem um único registro de GD**. O grão da Silver é 1 linha por UC.

In [0]:
n = gd.count()
print("linhas:                          ", n_br(n))
print("linhas distintas (sem auditoria):", n_br(gd.select(cols_origem).distinct().count()))
print("id_uc distintos:                 ", n_br(gd.select(COL_UC).distinct().count()))

# Se existir uma coluna de id do registro (ex.: id_gd), ela também entra aqui
ids_registro = [c for c in cols_origem if c.startswith("id_") and c != COL_UC]
for c in ids_registro:
    print(f"{c} distintos:", n_br(gd.select(c).distinct().count()))

display(gd.groupBy(COL_UC).count().filter("count > 1")
        .groupBy("count").agg(F.count("*").alias("qtd_ucs")).orderBy("count"))

In [0]:
# UCs com mais de uma versão: quais colunas mudam?
versoes = gd.select(cols_origem).distinct()
ucs_multi = versoes.groupBy(COL_UC).count().filter("count > 1").select(COL_UC)
print("UCs com versões diferentes:", n_br(ucs_multi.count()))

dif = versoes.join(ucs_multi, COL_UC)
display(dif.groupBy(COL_UC).agg(
    *[(F.countDistinct(F.coalesce(F.col(c), F.lit("<nulo>"))) > 1).cast("int").alias(c)
      for c in cols_origem if c != COL_UC]
).select([F.sum(c).alias(c) for c in cols_origem if c != COL_UC]))
display(dif.orderBy(COL_UC).limit(20))

## 6. Formatos

**O que explora:** o padrão de cada coluna (dígitos viram `9`, letras viram `A`). Mostra datas em formatos mistos, números com vírgula, ids fora do padrão, unidades escritas junto do número (`5,4 kWp`) etc.

**Achado:** `id_uc` sempre `UC` + 6 dígitos; `data_conexao` sempre ISO (`yyyy-MM-dd`); `energia_injetada_kwh_media_mensal` sempre com ponto decimal. Só `potencia_kwp` tem problemas: **316 valores com vírgula decimal** (`9,99`, `99,99`, `9999,9`, `9999,99`) e **35 valores com 4 dígitos antes da vírgula** (`9999.99`, `9999.9`, `9999,9`, `9999,99`), investigados nas seções 9 e 14.

In [0]:
for c in cols_origem:
    print(c)
    display(gd.groupBy(padrao(F.col(c)).alias("padrao")).count().orderBy(F.desc("count")).limit(10))

## 7. Categorias

**O que explora:** colunas com poucos valores distintos (até 30), como vieram e depois de padronizar. Se o número cai depois da padronização, há grafias diferentes para o mesmo valor (como `tipo` em medidores).

**Achado:** `modalidade` tem 3 valores, sem grafias diferentes: `GERACAO NA PROPRIA UC` (3.042), `AUTOCONSUMO REMOTO` (321) e `GERACAO COMPARTILHADA` (147).

In [0]:
for c in cols_origem:
    brutos = gd.select(c).distinct().count()
    if brutos <= 30:
        padronizados = gd.select(sem_acento(F.col(c))).distinct().count()
        print(f"{c}: {brutos} valores brutos -> {padronizados} depois de padronizar")
        display(gd.groupBy(c).count().orderBy(F.desc("count")))

## 8. Integridade com unidades consumidoras

**O que explora:**
- `id_uc` no padrão `UC` + 6 dígitos (órfãs de leituras e faturas tinham 7 dígitos).
- `id_uc` que não existe na `silver.unidades_consumidoras`.
- Se alguma das **200 UCs de teste** (que estão na Bronze de UCs, mas não na Silver) tem GD.

**Achado:** nenhuma órfã: todos os `id_uc` estão no padrão e existem na Silver de UCs. Nenhuma das 200 UCs de teste tem GD.

In [0]:
ucs_silver = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc")
ucs_bronze = (spark.table(f"{bronze_schema}.unidades_consumidoras")
              .filter(F.col("id_uc") != "id_uc").select("id_uc").distinct())
ucs_teste = ucs_bronze.join(ucs_silver, "id_uc", "left_anti")

uc = gd.withColumnRenamed(COL_UC, "id_uc")
print("id_uc fora do padrão UC + 6 dígitos:",
      n_br(uc.filter(~F.coalesce(F.col("id_uc").rlike("^UC[0-9]{6}$"), F.lit(False))).count()))
print("id_uc inexistente na Silver de UCs:  ", n_br(uc.join(ucs_silver, "id_uc", "left_anti").count()))
print("registros em UCs de teste:           ", n_br(uc.join(ucs_teste, "id_uc", "left_semi").count()))
display(uc.join(ucs_silver, "id_uc", "left_anti").limit(10))

## 9. Potência instalada

**O que explora:** se a potência converte para número (vírgula decimal, unidade escrita junto), valores zero, negativos ou absurdos, e a coerência com a classe da UC. Pela regulação, **microgeração** vai até 75 kW e **minigeração** de 75 kW a 5 MW: uma residência com centenas de kW é suspeita.

**Achado:** todas as potências convertem para número depois de trocar a vírgula por ponto, e nenhuma é zero ou negativa. A mediana é 5,87 kWp, mas **35 UCs têm entre 1.000 e 8.995 kWp** (33 residenciais e 2 comerciais). Uma residência com 9 mil kWp não existe (o limite da minigeração é 5.000 kW). A seção 14 confirma que esses valores foram registrados em **watts**.

In [0]:
pot = (uc.withColumn("pot_txt", F.regexp_replace(F.trim(F.col(COL_POTENCIA)), "(?i)\\s*kwp?$", ""))
          .withColumn("pot", F.expr("try_cast(regexp_replace(pot_txt, ',', '.') as double)")))

print("não converte para número:", n_br(pot.filter(F.col(COL_POTENCIA).isNotNull() & F.col("pot").isNull()).count()))
display(pot.filter(F.col(COL_POTENCIA).isNotNull() & F.col("pot").isNull()).groupBy(COL_POTENCIA).count().limit(20))
print("zero ou negativa:", n_br(pot.filter("pot <= 0").count()))
display(pot.select(F.min("pot"), F.expr("percentile_approx(pot, 0.5)").alias("mediana"),
                   F.expr("percentile_approx(pot, 0.99)").alias("p99"), F.max("pot")))

classes = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc", "classe", "municipio")
display(pot.join(classes, "id_uc", "left").groupBy("classe").agg(
    F.count("*").alias("registros"), F.round(F.avg("pot"), 2).alias("media_kwp"),
    F.max("pot").alias("max_kwp"), F.sum((F.col("pot") > 75).cast("int")).alias("acima_75kw")).orderBy("classe"))

## 10. Data de conexão

**O que explora:** formatos e ordem dia/mês, com a mesma técnica das outras tabelas.

- **Teste 1, partes maiores que 12:** se a 2ª parte passa de 12 no hífen, o hífen é mês primeiro (como em leituras, faturas e medidores).
- **Teste 2, coerência com a UC:** a GD não pode ser conectada antes da ligação da UC. Com a ordem certa, quase nenhuma data fica antes da ligação; com a ordem trocada, aparecem mais.
- **Datas impossíveis:** depois do fim da extração (2026-06-30) ou muito antigas (GD só existe no Brasil desde a REN 482, de 2012).

**Achado:** só ISO, todas convertem, nenhuma depois de 2026-06-30 nem antes de 2012. Conexões de **jan/2024 a jun/2026** (2024: 1.845; 2025: 1.304; 2026: 361). **5 UCs têm conexão antes da ligação da UC**, de 55 a 473 dias antes (seção 14).

In [0]:
dt = uc.withColumn("padrao_data", padrao(F.trim(F.col(COL_DATA))))
display(dt.groupBy("padrao_data").count().orderBy(F.desc("count")))

nao_iso = (dt.filter(F.col("padrao_data").isin("99-99-9999", "99.99.9999", "99/99/9999"))
           .withColumn("p1", F.substring(F.trim(F.col(COL_DATA)), 1, 2).cast("int"))
           .withColumn("p2", F.substring(F.trim(F.col(COL_DATA)), 4, 2).cast("int")))
display(nao_iso.groupBy("padrao_data").agg(
    F.count("*").alias("linhas"),
    F.sum((F.col("p1") > 12).cast("int")).alias("primeira_parte_maior_12"),
    F.sum((F.col("p2") > 12).cast("int")).alias("segunda_parte_maior_12")))

In [0]:
def converte(col, fmt):
    return F.expr(f"to_date(try_to_timestamp(trim({col}), '{fmt}'))")

ligacao = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc", "data_ligacao")
candidatos = {
    "9999-99-99": ["yyyy-MM-dd"],
    "99-99-9999": ["MM-dd-yyyy", "dd-MM-yyyy"],
    "99.99.9999": ["dd.MM.yyyy", "MM.dd.yyyy"],
    "99/99/9999": ["dd/MM/yyyy", "MM/dd/yyyy"],
}
dt_lig = dt.join(ligacao, "id_uc", "left")
for p, fmts in candidatos.items():
    sub = dt_lig.filter(F.col("padrao_data") == p)
    if sub.count() == 0:
        continue
    for fmt in fmts:
        conv = sub.withColumn("conexao", converte(COL_DATA, fmt))
        print(f"{p} como {fmt}: linhas {n_br(sub.count())} | não converte {n_br(conv.filter('conexao IS NULL').count())}"
              f" | antes da ligação da UC {n_br(conv.filter('conexao < data_ligacao').count())}")

In [0]:
# Conversão com a convenção do projeto (hífen = mês primeiro). Confira com os testes acima antes de confiar.
formatos = {"9999-99-99": "yyyy-MM-dd", "99-99-9999": "MM-dd-yyyy",
            "99.99.9999": "dd.MM.yyyy", "99/99/9999": "dd/MM/yyyy"}
conv = None
for p, fmt in formatos.items():
    cond, val = F.col("padrao_data") == p, converte(COL_DATA, fmt)
    conv = F.when(cond, val) if conv is None else conv.when(cond, val)

gd_dt = dt.withColumn("conexao", conv).withColumn("mes_conexao", F.date_format("conexao", "yyyy-MM"))
print("não converteu:", n_br(gd_dt.filter("conexao IS NULL").count()))
print("depois de 2026-06-30:", n_br(gd_dt.filter("conexao > DATE'2026-06-30'").count()))
print("antes de 2012:", n_br(gd_dt.filter("conexao < DATE'2012-01-01'").count()))
display(gd_dt.groupBy(F.year("conexao").alias("ano")).count().orderBy("ano"))

## 11. GD ao longo do tempo

**O que explora:** quantas UCs tinham GD conectada **até o fim do primeiro ano** (2023-06) e **até o fim dos últimos 12 meses** (2026-06), e a evolução mês a mês. É o número central da hipótese: se as UCs com GD cresceram muito entre os dois períodos, a GD explica parte da queda de volume.

Usa a **primeira** conexão de cada UC (se ela tiver mais de um registro).

**Achado:** **nenhuma UC tinha GD no primeiro ano**; no fim dos últimos 12 meses são **3.510 (7,8% da base)**. O ritmo de novas conexões cai de forma contínua: 183 em jan/2024, 49 em jun/2026. Só há GD em UCs **residenciais** (3.127) e **comerciais** (383), concentradas em Recife, Jaboatão e Olinda.

In [0]:
prim_conexao = gd_dt.groupBy("id_uc").agg(F.min("conexao").alias("conexao"))
prim_conexao = prim_conexao.withColumn("mes_conexao", F.date_format("conexao", "yyyy-MM"))

total_ucs = ucs_silver.count()
ate_p0 = prim_conexao.filter(F.col("mes_conexao") <= P0_FIM).count()
ate_p1 = prim_conexao.filter(F.col("mes_conexao") <= P1_FIM).count()
print(f"UCs com GD até {P0_FIM}: {n_br(ate_p0)} ({ate_p0 / total_ucs:.1%} da base)")
print(f"UCs com GD até {P1_FIM}: {n_br(ate_p1)} ({ate_p1 / total_ucs:.1%} da base)")

# Evolução acumulada por mês
por_mes = prim_conexao.groupBy("mes_conexao").count().orderBy("mes_conexao")
from pyspark.sql import Window
display(por_mes.withColumn("acumulado", F.sum("count").over(Window.orderBy("mes_conexao"))))

# Onde estão: município e classe
display(prim_conexao.join(classes, "id_uc", "left").groupBy("municipio", "classe").count().orderBy(F.desc("count")))

## 12. Impacto no volume faturado

**A pergunta do diretor:** o volume faturado caiu ~4% entre o primeiro ano e os últimos 12 meses. Quanto dessa queda vem das UCs com GD?

**Como:** separa as UCs em grupos pela data da primeira conexão e compara o volume de cada grupo nos dois períodos (`silver.leituras`, `consumo_kwh`).

| Grupo | Significado |
|---|---|
| `SEM_GD` | nunca teve GD |
| `GD_ANTES_P0` | conectou até o fim do primeiro ano (já tinha GD nos dois períodos) |
| `GD_ENTRE` | conectou entre os dois períodos (sem GD no primeiro ano, com GD no último) |
| `GD_DURANTE_P1` | conectou durante os últimos 12 meses (efeito parcial) |

Se a queda se concentra em `GD_ENTRE` e `GD_DURANTE_P1`, enquanto `SEM_GD` fica estável, a GD é uma causa forte. A coluna `participacao_na_variacao` mostra quanto da variação total cada grupo explica.

**Achado: a GD explica toda a queda de volume.**

| Grupo | Primeiro ano (kWh) | Últimos 12 meses (kWh) | Variação | Parte da variação total |
|---|---|---|---|---|
| `SEM_GD` | 208.754.426 | 210.977.944 | **+1,07%** | −24,8% |
| `GD_ENTRE` | 13.584.677 | 4.253.989 | −68,69% | 104,0% |
| `GD_DURANTE_P1` | 4.713.824 | 2.847.121 | −39,6% | 20,8% |
| **Total** | 227.052.927 | 218.079.054 | **−3,95%** | 100% |

- O total reproduz o **−4% do diretor** (−3,95%).
- **Sem a GD, o volume teria crescido 1,07%.** As UCs que conectaram GD perderam 11,2 milhões de kWh faturados, mais do que a queda total (9,0 milhões), porque o resto da base cresceu.
- **Antes × depois na mesma UC:** nas 2.587 UCs com 12 meses de leitura dos dois lados, o consumo faturado cai **69,6%** depois da conexão (mediana e média iguais, 100% das UCs com queda acima de 30%), tanto no residencial quanto no comercial. O efeito é praticamente constante entre as UCs.
- `GD_DURANTE_P1` cai menos (−39,6%) porque essas UCs só têm GD em parte dos últimos 12 meses.

In [0]:
leit = spark.table(f"{silver_schema}.leituras").select("id_uc", "mes_referencia", "consumo_kwh")
leit = leit.withColumn("periodo", F.when(F.col("mes_referencia").between(P0_INI, P0_FIM), "P0")
                                   .when(F.col("mes_referencia").between(P1_INI, P1_FIM), "P1")).filter("periodo IS NOT NULL")

grupo = (F.when(F.col("mes_conexao").isNull(), "SEM_GD")
          .when(F.col("mes_conexao") <= P0_FIM, "GD_ANTES_P0")
          .when(F.col("mes_conexao") < P1_INI, "GD_ENTRE")
          .otherwise("GD_DURANTE_P1"))
base = leit.join(prim_conexao.select("id_uc", "mes_conexao"), "id_uc", "left").withColumn("grupo", grupo)

vol = (base.groupBy("grupo").pivot("periodo", ["P0", "P1"]).agg(F.sum("consumo_kwh"))
       .withColumn("variacao_kwh", F.col("P1") - F.col("P0"))
       .withColumn("variacao_pct", F.round((F.col("P1") / F.col("P0") - 1) * 100, 2)))
total = vol.agg(F.sum("P0").alias("P0"), F.sum("P1").alias("P1")).first()
var_total = total["P1"] - total["P0"]
print(f"Volume total: P0 = {n_br(int(total['P0']))} kWh | P1 = {n_br(int(total['P1']))} kWh | "
      f"variação {var_total / total['P0']:.2%}  (o diretor fala em quase -4%)")
display(vol.withColumn("participacao_na_variacao_pct", F.round(F.col("variacao_kwh") / F.lit(var_total) * 100, 1))
        .orderBy("grupo"))

In [0]:
# Antes x depois na mesma UC: consumo médio nos 12 meses antes e depois da conexão
# (só UCs com 12 meses de leitura dos dois lados; o mês da conexão fica de fora)
lc = spark.table(f"{silver_schema}.leituras").select("id_uc", "mes_referencia", "consumo_kwh")
lc = lc.withColumn("mes", F.to_date(F.concat("mes_referencia", F.lit("-01"))))
pc = prim_conexao.withColumn("mes_con", F.to_date(F.concat("mes_conexao", F.lit("-01"))))

ab = (lc.join(pc.select("id_uc", "mes_con"), "id_uc")
        .withColumn("dist", F.months_between("mes", "mes_con"))
        .withColumn("lado", F.when(F.col("dist").between(-12, -1), "antes").when(F.col("dist").between(1, 12), "depois"))
        .filter("lado IS NOT NULL")
        .groupBy("id_uc").pivot("lado", ["antes", "depois"]).agg(F.avg("consumo_kwh").alias("media"), F.count("consumo_kwh").alias("n")))
ab = ab.filter("antes_n = 12 AND depois_n = 12").withColumn(
    "variacao_pct", (F.col("depois_media") / F.col("antes_media") - 1) * 100)

print("UCs com 12 meses antes e depois:", n_br(ab.count()))
display(ab.select(F.round(F.expr("percentile_approx(variacao_pct, 0.5)"), 1).alias("mediana_variacao_pct"),
                  F.round(F.avg("variacao_pct"), 1).alias("media_variacao_pct"),
                  F.sum((F.col("variacao_pct") < -30).cast("int")).alias("ucs_queda_acima_30pct")))
display(ab.join(classes, "id_uc", "left").groupBy("classe").agg(
    F.count("*").alias("ucs"), F.round(F.expr("percentile_approx(variacao_pct, 0.5)"), 1).alias("mediana_variacao_pct")))

## 14. Checagens complementares

Três perguntas que ficaram abertas nas seções 9, 10 e 12.

**Achados:**
- **Potência em watts:** nas 3.475 UCs normais, a energia injetada é de **33,8 kWh por kWp** por mês (mediana). Nas 35 suspeitas, esse número dá 0,03; **dividindo a potência por 1.000, dá 33,3**, o mesmo das normais. A energia injetada mediana também é parecida (187 contra 199 kWh). Conclusão: os 35 valores estão em **W**, e o valor certo é potência / 1.000 (de 1,5 a 9 kWp, como as outras).
- **Conexão antes da ligação:** 5 UCs, de 55 a 473 dias antes. Não há gabarito para saber qual das duas datas está errada, e o efeito é pequeno (5 de 3.510).

In [0]:
# Teste: as potências com 4 dígitos foram registradas em W?
t = (pot.withColumn("energia", F.expr("try_cast(energia_injetada_kwh_media_mensal as double)"))
        .withColumn("suspeita", F.col("pot") >= 1000)
        .withColumn("kwh_por_kwp", F.col("energia") / F.col("pot"))
        .withColumn("kwh_por_kwp_se_watts", F.col("energia") / (F.col("pot") / 1000)))
display(t.groupBy("suspeita").agg(
    F.count("*").alias("ucs"),
    F.round(F.expr("percentile_approx(kwh_por_kwp, 0.5)"), 1).alias("kwh_por_kwp"),
    F.round(F.expr("percentile_approx(kwh_por_kwp_se_watts, 0.5)"), 1).alias("kwh_por_kwp_se_fosse_watts"),
    F.round(F.expr("percentile_approx(energia, 0.5)"), 1).alias("energia_mediana")))

In [0]:
# Conexões antes da ligação da UC: quanto antes?
display(gd_dt.join(ligacao, "id_uc")
        .filter("conexao < data_ligacao")
        .select("id_uc", "conexao", "data_ligacao", F.datediff("data_ligacao", "conexao").alias("dias_antes")))

## 15. Resumo e regras para a Silver

| Problema | Linhas | Regra | Log |
|---|---|---|---|
| `potencia_kwp` com vírgula decimal | 316 | trocar vírgula por ponto e converter para `decimal` | G1 |
| `potencia_kwp` registrada em watts | 35 | dividir por 1.000 + flag `potencia_convertida_de_w` | G2 |
| `data_conexao` antes da ligação da UC | 5 | manter a data + flag `conexao_antes_ligacao` | G3 |
| Tipos | todas | `data_conexao` → `date`; potência e energia → `decimal` | G4 |

**Verificado e sem problema:** sem cabeçalho repetido, cópias, nulos, códigos de ausência, órfãs ou UCs de teste; 1 registro por UC; datas só em ISO; `modalidade` com 3 valores limpos.

**Para o diagnóstico:**
- 0 UCs com GD no primeiro ano → **3.510 (7,8% da base)** nos últimos 12 meses, todas conectadas de 2024 em diante.
- As UCs com GD explicam **mais de 100% da queda de volume**: sem elas, o volume teria crescido 1,07%.
- Depois da conexão, o consumo faturado da UC cai **69,6%**.
- O ritmo de novas conexões está caindo (183/mês no início de 2024; 49/mês em jun/2026).